# 🧪 PumpGatewayWS

## 📦 Imports y Configuración Inicial


In [ ]:
import os
from dotenv import load_dotenv

from logging_system import setup_logging
from pumpfun.redis_bridge import (
    RedisBridgeConfig,
    run_service
)

In [ ]:
load_dotenv()

CONSOLE_OUTPUT = os.getenv("CONSOLE_OUTPUT", "true") == "true"
FILE_OUTPUT = os.getenv("FILE_OUTPUT", "false") == "true"
MIN_LEVEL_TO_PROCESS = os.getenv("MIN_LEVEL_TO_PROCESS", "DEBUG")
ENABLE_LOGFIRE = os.getenv("ENABLE_LOGFIRE", "false") == "true"
LOGFIRE_MIN_LEVEL = os.getenv("LOGFIRE_MIN_LEVEL", "WARNING")
ENVIRONMENT = os.getenv("ENVIRONMENT", "development")

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379/0")
NAMESPACE = os.getenv("NAMESPACE", "pumpfun")
API_KEY = os.getenv("PUMPFUN_API_KEY")
INACTIVITY_WATCH_SECONDS = int(os.getenv("INACTIVITY_WATCH_SECONDS", 600))
WEBSOCKET_TIMEOUT = int(os.getenv("WEBSOCKET_TIMEOUT", 60))

setup_logging(
    console_output=CONSOLE_OUTPUT,
    file_output=FILE_OUTPUT,
    log_directory="copy_trading/logs",
    log_filename="pumpfun_redis_bridge_%Y-%m-%d_%H-%M-%S.log",
    min_level_to_process=MIN_LEVEL_TO_PROCESS, # type: ignore
    enable_logfire=ENABLE_LOGFIRE,
    logfire_config={
        "service_name": "pumpfun_redis_bridge",
        "environment": ENVIRONMENT,
        "tags": {
            "project": "pumpfun-copy-trading-system",
            "version": "1.0.0"
        }
    },
    logfire_min_level=LOGFIRE_MIN_LEVEL # type: ignore
)

CONFIG = RedisBridgeConfig(
    redis_url=REDIS_URL,
    namespace=NAMESPACE,
    api_key=API_KEY,
    inactivity_watch_seconds=INACTIVITY_WATCH_SECONDS,
    websocket_timeout=WEBSOCKET_TIMEOUT,
)

print(f"✅ Configuración creada:")
print(f"   - Redis URL: {CONFIG.redis_url}")
print(f"   - Namespace: {CONFIG.namespace}")
print(f"   - API Key configurada: {'✓' if CONFIG.api_key else '✗'}")


## 🚀 Iniciar el Servicio Bridge

El servicio se iniciará en background y procesará:
- Comandos entrantes desde el canal `pumpfun:commands`
- Eventos desde PumpFun WebSocket
- Redistribución a clientes vía canales individuales


In [ ]:
await run_service(CONFIG)


## 📝 Notas Adicionales

### Canales Redis utilizados:
- **Comandos**: `{namespace}:commands` (shared)
- **Eventos**: `{namespace}:events:{client_id}` (por cliente)
- **Respuestas**: `{namespace}:responses:{client_id}` (por cliente)

### Comandos soportados:
1. `subscribe_account_trade`: Suscribirse a wallets
2. `unsubscribe_account_trade`: Desuscribirse de wallets específicas
3. `unsubscribe_all`: Desuscribirse de todas las wallets
4. `ping`: Verificar conectividad

### Optimizaciones del Bridge:
- ✅ Deduplicación de suscripciones a PumpFun
- ✅ Liberación automática cuando no hay clientes
- ✅ Gestión de múltiples clientes concurrentes
- ✅ Manejo de errores y comandos inválidos